#Git 관련 코드

In [2]:
!pwd
%cd /content/drive/MyDrive/Project/Sentence-generator-using-LSTM

/content
/content/drive/MyDrive/Project/Sentence-generator-using-LSTM


In [3]:
!git add .

In [5]:
!git config --global user.email "rudeore0928@gmail.com"
!git config --global user.name "rudeore-098"

!git commit -m "Seq2Seq Architecture"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Sentence-generator-using-LSTM.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


In [ ]:
!git reset --soft HEAD~1

In [ ]:
!git add .
!git commit -m "TEST"

In [ ]:
!git push

Enumerating objects: 5, done.


#데이터처리

In [21]:
import pandas as pd

# GitHub의 Raw 데이터 주소
train_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
test_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

# 데이터 불러오기
train_df = pd.read_csv(train_url, sep='\t')
test_df = pd.read_csv(test_url, sep='\t')

# 데이터 확인
print(f"훈련 데이터 크기: {train_df.shape}")
print(f"테스트 데이터 크기: {test_df.shape}")
print(train_df.head())

#불필요한 데이터 제거
texts = train_df['document'].dropna().values

훈련 데이터 크기: (150000, 3)
테스트 데이터 크기: (50000, 3)
         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1


In [22]:
!pip install konlpy

In [23]:
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
from konlpy.tag import Mecab # Mecab 도입 추천

# 전처리 함수
def clean_text(text):
    # 특수문자 제거 (한글, 영어, 숫자, 기본 문장부호 제외)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.?!,]', ' ', str(text))

    # 특수한 input 제거
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 토크나이저
try:
    mecab = Mecab()
    def tokenizer(text):
        return mecab.morphs(clean_text(text))
except:
    def tokenizer(text):
        return clean_text(text).split()

#  Vocab 구축
special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
all_tokens = []

# 전처리가 적용된 상태로 토큰 추출
for text in texts[:10000]:
    all_tokens.extend(tokenizer(text))

# 빈도수 필터링, 최소 빈도 2
MIN_FREQ = 2
counts = Counter(all_tokens)
vocab = special_tokens + [word for word, count in counts.most_common() if count >= MIN_FREQ]

# 매핑 딕셔너리 생성
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

# 주요 인덱스 상수화
PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

In [24]:

lengths = [len(tokenizer(clean_text(t))) for t in texts[:10000]]
print(f"평균 길이: {sum(lengths)/len(lengths)}")
print(f"최대 길이: {max(lengths)}")

# data set의 길이 설정
import numpy as np
max_len = int(np.percentile(lengths, 95))
print(f"권장 max_len (95% 커버): {max_len}")
print(len(word2idx))
print(len(idx2word))
print(word2idx)

평균 길이: 7.719
최대 길이: 38
권장 max_len (95% 커버): 23
7173
7173
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, '영화': 4, '정말': 5, '너무': 6, '진짜': 7, '영화.': 8, '이': 9, '.': 10, '그냥': 11, '이런': 12, '왜': 13, '더': 14, '다': 15, '영화를': 16, '잘': 17, '보고': 18, '수': 19, '영화가': 20, '그': 21, '본': 22, '좀': 23, '이렇게': 24, '최고의': 25, '없는': 26, '영화는': 27, '이건': 28, '이거': 29, '평점': 30, '내가': 31, '있는': 32, '역시': 33, '완전': 34, '다시': 35, '평점이': 36, '이게': 37, '참': 38, '보는': 39, '좋은': 40, '연기': 41, '난': 42, '봤는데': 43, '한': 44, '아': 45, '내': 46, '하는': 47, '그리고': 48, '꼭': 49, '가장': 50, '또': 51, '많이': 52, '없다.': 53, '것': 54, '없고': 55, '쓰레기': 56, '드라마': 57, '보면': 58, '재밌게': 59, '같은': 60, '솔직히': 61, '10점': 62, '봐도': 63, '스토리': 64, '최고': 65, '영화의': 66, '대한': 67, '!': 68, '무슨': 69, '마지막': 70, '끝까지': 71, '전혀': 72, '하지만': 73, '넘': 74, '아주': 75, '뭔가': 76, '내내': 77, '뭐': 78, '안': 79, '감독의': 80, '별': 81, '볼': 82, '연기도': 83, '아직도': 84, '말이': 85, '지금': 86, '최악의': 87, '내용도': 88, '별로': 89, '하고': 90, '재미도': 91, '만든': 92, '어떻게': 9

In [25]:
class Seq2SeqDataset(Dataset):
    def __init__(self, texts, word2idx, max_len=23):
        self.texts = [str(t) for t in texts]
        self.word2idx = word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def encode(self, text, max_len, is_target=False):
        tokens = tokenizer(text)
        # 단어를 인덱스로, 모르면 UNK
        indices = [self.word2idx.get(t, UNK_IDX) for t in tokens]

        if is_target:
            # 출력용: <SOS> + 내용 + <EOS> (최대 20자 + 특수토큰 2개)
            indices = [SOS_IDX] + indices[:max_len] + [EOS_IDX]
        else:
            # 입력용: 내용 (최대 10자)
            indices = indices[:max_len]

        # Padding
        if len(indices) < (max_len + 2 if is_target else max_len):
            pad_len = (max_len + 2 if is_target else max_len) - len(indices)
            indices += [PAD_IDX] * pad_len
        else:
            indices = indices[:(max_len + 2 if is_target else max_len)]

        return torch.tensor(indices)

    def __getitem__(self, i):
        src = self.encode(self.texts[i], self.max_len, is_target=False)
        trg = self.encode(self.texts[i], self.max_len, is_target=True)
        return src, trg

# 데이터로더 생성
dataset = Seq2SeqDataset(texts[0:], word2idx)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [26]:
import torch
import torch.nn as nn
import random

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.n_layers = n_layers
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, bidirectional=True, batch_first=True)
        # Bi-LSTM(2*hid)을 Decoder(1*hid)용으로 맞추기 위한 Linear
        self.fc_hidden = nn.Linear(hid_dim * 2, hid_dim)
        self.fc_cell = nn.Linear(hid_dim * 2, hid_dim)

    def forward(self, src):
        outputs, (hidden, cell) = self.rnn(self.embedding(src))

        # hidden: [n_layers*2, batch, hid_dim] -> [n_layers, batch, hid_dim]
        # 마지막 layer의 양방향 상태를 합쳐서 전달
        # hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))).unsqueeze(0)
        # cell = torch.tanh(self.fc_cell(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1))).unsqueeze(0)

        h_combined = torch.tanh(self.fc_hidden(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        c_combined = torch.tanh(self.fc_cell(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)))

        # 핵심: [1, batch, hid_dim]으로 만든 후, n_layers(2)만큼 복사하여 [2, batch, hid_dim] 생성
        hidden = h_combined.unsqueeze(0).repeat(self.n_layers, 1, 1)
        cell = c_combined.unsqueeze(0).repeat(self.n_layers, 1, 1)

        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(hid_dim, output_dim)

    def forward(self, input, hidden, cell):
        # input: [batch] -> [batch, 1]
        input = input.unsqueeze(1)
        embedded = self.embedding(input)
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(1))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        # 첫 번째 입력은 <SOS>
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

In [27]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


INPUT_DIM = len(word2idx)  # 단어 사전 크기
OUTPUT_DIM = len(word2idx) # 생성용 단어 사전 크기
ENC_EMB_DIM = 256          # 임베딩 차원
DEC_EMB_DIM = 256
HID_DIM = 512              # LSTM 은닉 상태 크기
N_LAYERS = 2               # 층 개수 (단순화를 위해 1층 권장, 발표 시 효율성 강조)
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

# 모델 인스턴스화
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

# 옵티마이저 및 손실 함수
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=word2idx['<PAD>'])

In [28]:
import time
def train(model, loader, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(loader):
        start_time = time.time()
        src, trg = src.to(device), trg.to(device)

        optimizer.zero_grad()

        # 모델 예측 (Teacher Forcing 0.5 적용)
        output = model(src, trg)

        # output: [batch, trg_len, vocab_size] -> [batch * trg_len, vocab_size]
        # trg: [batch, trg_len] -> [batch * trg_len]
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()
        end_time = time.time() # 에폭 종료 시간 기록
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    return epoch_loss / len(loader), epoch_mins, epoch_secs


# 소요 시간을 분/초로 변환해주는 유틸리티 함수
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

N_EPOCHS = 30
CLIP = 1

for epoch in range(N_EPOCHS):
    train_loss,epoch_mins, epoch_secs = train(model, loader, optimizer, criterion, CLIP)
    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Time: {epoch_mins}m {epoch_secs}s')

Epoch: 01 | Train Loss: 3.755 | Time: 0m 0s
Epoch: 02 | Train Loss: 3.200 | Time: 0m 0s
Epoch: 03 | Train Loss: 2.860 | Time: 0m 0s
Epoch: 04 | Train Loss: 2.674 | Time: 0m 0s
Epoch: 05 | Train Loss: 2.512 | Time: 0m 0s
Epoch: 06 | Train Loss: 2.354 | Time: 0m 0s
Epoch: 07 | Train Loss: 2.205 | Time: 0m 0s
Epoch: 08 | Train Loss: 2.073 | Time: 0m 0s
Epoch: 09 | Train Loss: 1.947 | Time: 0m 0s
Epoch: 10 | Train Loss: 1.836 | Time: 0m 0s
Epoch: 11 | Train Loss: 1.734 | Time: 0m 0s
Epoch: 12 | Train Loss: 1.643 | Time: 0m 0s
Epoch: 13 | Train Loss: 1.557 | Time: 0m 0s
Epoch: 14 | Train Loss: 1.482 | Time: 0m 0s
Epoch: 15 | Train Loss: 1.411 | Time: 0m 0s
Epoch: 16 | Train Loss: 1.343 | Time: 0m 0s
Epoch: 17 | Train Loss: 1.282 | Time: 0m 0s
Epoch: 18 | Train Loss: 1.222 | Time: 0m 0s
Epoch: 19 | Train Loss: 1.171 | Time: 0m 0s
Epoch: 20 | Train Loss: 1.118 | Time: 0m 0s
Epoch: 21 | Train Loss: 1.073 | Time: 0m 0s
Epoch: 22 | Train Loss: 1.031 | Time: 0m 0s
Epoch: 23 | Train Loss: 0.993 | 

In [29]:
def generate_sentence(model, src, word2idx, idx2word, device, max_len=20, min_len=5):
    model.eval()
    with torch.no_grad():
        src = src.to(device).unsqueeze(0)
        hidden, cell = model.encoder(src)

        input_idx = torch.tensor([word2idx['<SOS>']]).to(device)
        result_indices = []

        # 제외하고 싶은 토큰 인덱스들
        forbidden_indices = [word2idx['<UNK>'], word2idx['<PAD>'], word2idx['<SOS>']]

        for t in range(max_len):
            output, hidden, cell = model.decoder(input_idx, hidden, cell)

            for idx in set(result_indices):
                output[0, idx] -= 2.0  # 이미 나온 단어의 확률(Logit)을 낮춤 (숫자가 클수록 강력함)

            # <UNK>, <PAD>, <SOS>의 확률을 매우 낮게 강제 조정
            for idx in forbidden_indices:
                output[0, idx] = -1e10

            top1 = output.argmax(1)
            predicted_idx = top1.item()

            if predicted_idx == word2idx['<EOS>']:
                if len(result_indices) < min_len:
                    # <EOS>도 일단 제외하고 다음 순위 선택
                    output[0, word2idx['<EOS>']] = -1e10
                    predicted_idx = output.argmax(1).item()
                else:
                    break

            result_indices.append(predicted_idx)
            input_idx = torch.tensor([predicted_idx]).to(device)

        result_sentence = [idx2word[i] for i in result_indices]
        return " ".join(result_sentence)

In [ ]:
def predict_sentence(sentence, model, word2idx, idx2word, device):
    model.eval()

    tokens = tokenizer(sentence)
    indices = [word2idx.get(t, word2idx['<UNK>']) for t in tokens]

    indices = indices[:10]
    if len(indices) < 10:
        indices += [word2idx['<PAD>']] * (10 - len(indices))

    src_tensor = torch.LongTensor(indices).to(device)

    # 문장 생성 함수 호출
    # max_len=20, min_len=5 조건을 여기서 제어
    generated_text = generate_sentence(model, src_tensor, word2idx, idx2word, device, max_len=20, min_len=5)

    return generated_text

# --- 발표 시연용 루프 ---
def run_demo():
    print("=== Naver Movie 리뷰 생성기 시연 ===")
    while True:
        input_str = input("리뷰 시작 문구 입력 (종료: q): ")
        if input_str.lower() == 'q': break

        output = predict_sentence(input_str, model, word2idx, idx2word, device)
        print(f"\n[Input]  : {input_str}")
        print(f"[Output] : {output}\n")
        print("-" * 40)
run_demo()

=== Naver Movie 리뷰 생성기 시연 ===
리뷰 시작 문구 입력 (종료: q): 영화

[Input]  : 영화
[Output] : 영화 너무 최고. 최고. 추천합니다

----------------------------------------
리뷰 시작 문구 입력 (종료: q): 별로

[Input]  : 별로
[Output] : 별로 안 하네요 최고. ,.

----------------------------------------
리뷰 시작 문구 입력 (종료: q): 배고파

[Input]  : 배고파
[Output] : ? 영화 ,. 최고. ,.

----------------------------------------
리뷰 시작 문구 입력 (종료: q): 재미

[Input]  : 재미
[Output] : 재미 ! 같다 최고 10자

----------------------------------------
리뷰 시작 문구 입력 (종료: q): 10자

[Input]  : 10자
[Output] : 10자 . 추천 10자 노잼

----------------------------------------
리뷰 시작 문구 입력 (종료: q): 추천

[Input]  : 추천
[Output] : 추천 최고 추천 ! 좋았음

----------------------------------------
리뷰 시작 문구 입력 (종료: q): ㅋㅋㅋㅋ

[Input]  : ㅋㅋㅋㅋ
[Output] : ? 최고. ! 0 추천.

----------------------------------------


NameError: name 'torch' is not defined